# 05. Запасы, риски и бизнес-выводы

В исходных данных нет фактических складских остатков. Поэтому расчеты ниже — сценарный анализ на основе прогноза спроса, ошибки прогноза и простого бизнес-допущения `lead_time_days = 7`. Это не доказанный экономический эффект, а способ выделить товары и рынки для контроля.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.baselines import (
    median_by_weekday,
    moving_average_7,
    moving_average_28,
    naive_last_value,
    seasonal_naive_7,
    seasonal_naive_28,
)
from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.inventory import inventory_policy_table
from src.validation import train_holdout_split

## Модельный прогноз

In [2]:
lead_time_days = 7
forecast_path = RESULTS_DIR / 'forecast_vs_actual_sample.csv'
if not forecast_path.exists():
    raise FileNotFoundError('Сначала запустите notebook 04, чтобы получить forecast_vs_actual_sample.csv')

model_forecast = pd.read_csv(forecast_path, parse_dates=['sales_date'])
model_policy = inventory_policy_table(model_forecast, lead_time_days=lead_time_days)
model_policy['source'] = 'model'
model_policy.head()

,stock_code,market_id,observed_days,avg_daily_demand,demand_std,avg_forecast_sales,forecast_error,forecast_bias,stockout_risk_rate,overstock_risk_rate,stockout_risk_flag,overstock_risk_flag,negative_actual_sales_count,lead_time_days,z_score,available_stock_assumption,suggested_safety_stock,suggested_reorder_point,source
0,22053,United Kingdom,1,1.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,7.0,model
1,21913,France,1,4.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,28.0,model
2,21485,Austria,1,3.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,21.0,model
3,16207A,United Kingdom,1,1.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,7.0,model
4,21847,United Kingdom,1,1.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,7.0,model


## Baseline для сравнения рисков

In [3]:
baseline_functions = {
    'naive_last_value': naive_last_value,
    'seasonal_naive_7': seasonal_naive_7,
    'seasonal_naive_28': seasonal_naive_28,
    'moving_average_7': moving_average_7,
    'moving_average_28': moving_average_28,
    'median_by_weekday': median_by_weekday,
}

baseline_metrics_path = RESULTS_DIR / 'baseline_metrics.csv'
if baseline_metrics_path.exists():
    baseline_metrics = pd.read_csv(baseline_metrics_path)
    best_baseline = (
        baseline_metrics[baseline_metrics['metric'].str.lower() == 'wmape']
        .sort_values('value')
        .iloc[0]['baseline']
    )
else:
    best_baseline = 'moving_average_7'

features_path = PROCESSED_DATA_DIR / 'features_lags_rolling.csv'
features = pd.read_csv(features_path, parse_dates=['sales_date'])
baseline_frame = features[['sales_date', 'stock_code', 'market_id', 'net_sales_qty']].copy()
baseline_frame['forecast_net_sales_qty'] = baseline_functions[best_baseline](baseline_frame)
_, baseline_holdout = train_holdout_split(baseline_frame.dropna(subset=['forecast_net_sales_qty']), 'sales_date', holdout_days=28)
baseline_policy = inventory_policy_table(baseline_holdout, lead_time_days=lead_time_days)
baseline_policy['source'] = best_baseline
best_baseline, baseline_policy.head()

C:\Users\Ahmed The Best\AppData\Local\Temp\ipykernel_13676\1966108475.py:22: DtypeWarning: Columns (0: stock_code) have mixed types. Specify dtype option on import or set low_memory=False.
  features = pd.read_csv(features_path, parse_dates=['sales_date'])


('median_by_weekday',
   stock_code       market_id  observed_days  avg_daily_demand  demand_std  \
 0      22211           Spain              1              12.0         0.0   
 1          M         Germany              2               1.0         0.0   
 2      23206       Australia              1             100.0         0.0   
 3      23160           Spain              1              24.0         0.0   
 4     90026D  United Kingdom              1               1.0         0.0   
 
    avg_forecast_sales  forecast_error  forecast_bias  stockout_risk_rate  \
 0                 0.0             1.0           -1.0                 1.0   
 1                 0.0             1.0           -1.0                 1.0   
 2                 0.0             1.0           -1.0                 1.0   
 3                 0.0             1.0           -1.0                 1.0   
 4                 0.0             1.0           -1.0                 1.0   
 
    overstock_risk_rate  stockout_risk_flag 

## Таблица рисков

In [4]:
inventory_risk_table = pd.concat([model_policy, baseline_policy], ignore_index=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
inventory_risk_table.to_csv(RESULTS_DIR / 'inventory_risk_table.csv', index=False)
inventory_risk_table.head()

,stock_code,market_id,observed_days,avg_daily_demand,demand_std,avg_forecast_sales,forecast_error,forecast_bias,stockout_risk_rate,overstock_risk_rate,stockout_risk_flag,overstock_risk_flag,negative_actual_sales_count,lead_time_days,z_score,available_stock_assumption,suggested_safety_stock,suggested_reorder_point,source
0,22053,United Kingdom,1,1.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,7.0,model
1,21913,France,1,4.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,28.0,model
2,21485,Austria,1,3.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,21.0,model
3,16207A,United Kingdom,1,1.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,7.0,model
4,21847,United Kingdom,1,1.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,7.0,model


## Top-20 с высоким сценарным stockout risk

In [5]:
top_stockout = (
    model_policy.sort_values(['stockout_risk_rate', 'forecast_error', 'avg_daily_demand'], ascending=False)
    .head(20)
)
top_stockout.to_csv(RESULTS_DIR / 'top_stockout_risk_products.csv', index=False)
top_stockout

,stock_code,market_id,observed_days,avg_daily_demand,demand_std,avg_forecast_sales,forecast_error,forecast_bias,stockout_risk_rate,overstock_risk_rate,stockout_risk_flag,overstock_risk_flag,negative_actual_sales_count,lead_time_days,z_score,available_stock_assumption,suggested_safety_stock,suggested_reorder_point,source
32,22393,Netherlands,1,96.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,672.0,model
16,16048,Italy,1,24.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,168.0,model
31,22596,Germany,1,24.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,168.0,model
34,84820,EIRE,1,16.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,112.0,model
11,37448,Italy,1,12.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,84.0,model
14,22567,Italy,1,12.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,84.0,model
23,84228,United Kingdom,1,12.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,84.0,model
29,21109,Portugal,1,12.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,84.0,model
35,22442,EIRE,1,12.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,84.0,model
28,21115,Portugal,1,8.0,0.0,0.0,1.0,-1.0,1.0,0.0,True,False,0,7,1.65,0.0,0.0,56.0,model


## Top-20 с риском избыточного запаса

In [6]:
top_overstock = (
    model_policy.sort_values(['overstock_risk_rate', 'forecast_bias', 'avg_forecast_sales'], ascending=False)
    .head(20)
)
top_overstock.to_csv(RESULTS_DIR / 'top_overstock_risk_products.csv', index=False)
top_overstock

,stock_code,market_id,observed_days,avg_daily_demand,demand_std,avg_forecast_sales,forecast_error,forecast_bias,stockout_risk_rate,overstock_risk_rate,stockout_risk_flag,overstock_risk_flag,negative_actual_sales_count,lead_time_days,z_score,available_stock_assumption,suggested_safety_stock,suggested_reorder_point,source
3531,23046,EIRE,1,2.000000,0.000000,147.364530,72.682265,72.682265,0.0,1.0,False,True,0,7,1.65,147.364530,0.000000,14.000000,model
3532,22754,Spain,1,1.000000,0.000000,71.306288,70.306288,70.306288,0.0,1.0,False,True,0,7,1.65,71.306288,0.000000,7.000000,model
3533,22753,Spain,1,1.000000,0.000000,70.892047,69.892047,69.892047,0.0,1.0,False,True,0,7,1.65,70.892047,0.000000,7.000000,model
3534,22464,Spain,1,1.000000,0.000000,60.086235,59.086235,59.086235,0.0,1.0,False,True,0,7,1.65,60.086235,0.000000,7.000000,model
3535,22034,United Kingdom,3,0.666667,0.577350,35.540265,52.310397,52.310397,0.0,1.0,False,True,1,7,1.65,35.540265,2.520417,7.187083,model
3536,22755,Spain,1,1.000000,0.000000,49.286430,48.286430,48.286430,0.0,1.0,False,True,0,7,1.65,49.286430,0.000000,7.000000,model
3537,22097,Germany,1,1.000000,0.000000,36.559157,35.559157,35.559157,0.0,1.0,False,True,0,7,1.65,36.559157,0.000000,7.000000,model
3538,22465,Spain,1,1.000000,0.000000,35.744900,34.744900,34.744900,0.0,1.0,False,True,0,7,1.65,35.744900,0.000000,7.000000,model
3539,23702,United Kingdom,2,1.500000,0.707107,53.557644,34.705096,34.705096,0.0,1.0,False,True,0,7,1.65,53.557644,3.086867,13.586867,model
3540,22132,Portugal,1,1.000000,0.000000,35.149566,34.149566,34.149566,0.0,1.0,False,True,0,7,1.65,35.149566,0.000000,7.000000,model


## Сравнение модели и baseline по рискам

In [7]:
risk_comparison = (
    inventory_risk_table.groupby('source', as_index=False)
    .agg(
        avg_stockout_risk_rate=('stockout_risk_rate', 'mean'),
        avg_overstock_risk_rate=('overstock_risk_rate', 'mean'),
        avg_forecast_error=('forecast_error', 'mean'),
        avg_forecast_bias=('forecast_bias', 'mean'),
    )
)
risk_comparison

,source,avg_stockout_risk_rate,avg_overstock_risk_rate,avg_forecast_error,avg_forecast_bias
0,median_by_weekday,0.424073,0.454804,1.892318,1.172566
1,model,0.248230,0.748436,1.459056,1.149573


## Выводы

- Расчеты показывают сценарный риск, а не фактический складской дефицит или излишек.
- Товары и рынки с максимальным stockout risk: `[A]`.
- Товары и рынки с максимальным overstock risk: `[B]`.
- Для групп с высоким `forecast_bias` стоит отдельно проверить систематическое завышение или занижение прогноза.
- Предложенное правило reorder point основано на `lead_time_days = 7`; при реальных сроках поставки расчет нужно пересчитать.